In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import matplotlib
import pingouin as pg
matplotlib.style.use('paper_style.mplstyle')

In [ ]:
def _resolve_overlaps(df: pd.DataFrame) -> pd.DataFrame:
    """Sort by start; merge same-behavior overlaps, clip different-behavior overlaps."""
    df = df.sort_values("start").reset_index(drop=True)
    result = []
    i = 0
    while i < len(df):
        cur = df.iloc[i].copy()
        while i + 1 < len(df):
            nxt = df.iloc[i + 1]
            if nxt["start"] >= cur["end"]:
                break                                      # no overlap
            if nxt["Behavior"] == cur["Behavior"]:
                cur["end"] = max(cur["end"], nxt["end"])  # merge
                i += 1
            else:
                cur["end"] = nxt["start"]                 # clip
                break
        if cur["end"] > cur["start"]:                     # drop zero-length
            result.append(cur)
        i += 1
    return pd.DataFrame(result).reset_index(drop=True)


In [ ]:

# ── Intervals for this animal/session ─────────────────────────────────────────
behavior_df = pd.read_csv('/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv', index_col=0)
behavior_df["time"] = pd.to_timedelta(behavior_df["time"])


records = []
for (animal, session, behavior, btype), grp in behavior_df.groupby(
    ["animal", "session", "Behavior", "behavior_type"]
):
    starts = grp[grp["Event_Type"] == "State start"]["time"].dt.total_seconds().to_numpy()
    stops  = grp[grp["Event_Type"] == "State stop" ]["time"].dt.total_seconds().to_numpy()
    for s, e in zip(starts, stops):
        records.append(dict(animal=animal, session=session,
                            Behavior=behavior, behavior_type=btype,
                            start=s, end=e))

intervals = (
    pd.DataFrame(records)
    .groupby(["animal", "session"], group_keys=False)
    .apply(_resolve_overlaps)
    .reset_index(drop=True)
)

print('these mice will be excluded from aggressive...')
print(intervals[
    (intervals["behavior_type"] == "Aggressive") &
    (intervals["animal"].isin(["C2F1", "C2M4", "C1M2"]))
])
EXCLUDE_AGG = ["C2F1", "C2M4", "C1M2"]   # <2 s, single bout, one session

agg_animals = [a for a in intervals.loc[intervals["behavior_type"] == "Aggressive", "animal"].unique()
               if a not in EXCLUDE_AGG]

In [ ]:
intervals["duration"] = intervals["end"] - intervals["start"]

# ── Total time = sum of all interval durations, per animal/session ────────────
total_time = (
    intervals
    .groupby(["animal", "session"])["duration"]
    .sum()
    .rename("total_duration")
)

# ── Sum duration per animal / session / behavior_type ─────────────────────────
duration_summary = (
    intervals
    .groupby(["animal", "session", "behavior_type"])["duration"]
    .sum()
    .reset_index()
)

duration_summary = duration_summary.join(total_time, on=["animal", "session"])
duration_summary["proportion"] = duration_summary["duration"] / duration_summary["total_duration"]
duration_summary['sex']=duration_summary['animal'].str[2]

In [ ]:
duration_summary

In [ ]:
duration_summary["sex_session"] = duration_summary["sex"] + "_" + duration_summary["session"]

hue_order = ["F_R1", "F_R2", "M_R1", "M_R2"]
# with 4 hues seaborn places groups at -0.3, -0.1, +0.1, +0.3
hue_xoff = {h: -0.3 + i * 0.2 for i, h in enumerate(hue_order)}

fig, ax = plt.subplots(figsize=(10, 5))

sb.barplot(data=duration_summary, x='behavior_type', y='proportion', hue='sex_session',
           hue_order=hue_order, alpha=0.5, ax=ax, errorbar='se', capsize=.1)
sb.stripplot(data=duration_summary, x='behavior_type', y='proportion', hue='sex_session',
             hue_order=hue_order, dodge=True, jitter=0, ax=ax, edgecolor='k', linewidth=1)

btype_order = [t.get_text() for t in ax.get_xticklabels()]
btype_to_x  = {bt: i for i, bt in enumerate(btype_order)}

for animal, animal_grp in duration_summary.groupby("animal"):
    sex = animal_grp["sex"].iloc[0]
    for bt, bt_grp in animal_grp.groupby("behavior_type"):
        r1 = bt_grp[bt_grp["session"] == "R1"]["proportion"].values
        r2 = bt_grp[bt_grp["session"] == "R2"]["proportion"].values
        if len(r1) and len(r2):
            x = btype_to_x[bt]
            ax.plot(
                [x + hue_xoff[f"{sex}_R1"], x + hue_xoff[f"{sex}_R2"]],
                [r1[0], r2[0]],
                color="gray", alpha=0.5, linewidth=0.8, zorder=0
            )

ax.set_xlabel("")
ax.legend(title="", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()


# Everybody proportion time interacting by sex

In [ ]:
BTYPES = ["Social Avoidance", "Pro-Social", "Aggressive"]
DODGE  = 0.2
session_xoff = {"R1": -DODGE, "R2": DODGE}
sex_order = ["F", "M"]
sex_to_x  = {s: i for i, s in enumerate(sex_order)}
agg_palette   = ["#fe8382","#fe0806",]
social_palette   = ["#aacffd","#55a0fb"]
avoidance_palette    = ["#dbaaea","#b856d7"]
palettes = [avoidance_palette,social_palette,agg_palette]
height=1.5
width=4

fig, axes = plt.subplots(nrows=1, ncols=3, dpi=200, figsize=(width, height), sharey=False)

for i,(ax, btype) in enumerate(zip(axes, BTYPES)):
    sub = duration_summary[duration_summary["behavior_type"] == btype]
    
    if btype == "Aggressive":
        sub = sub[~sub["animal"].isin(EXCLUDE_AGG)]
    sb.barplot(data=sub, x='sex', y='proportion', hue='session', hue_order=["R1", "R2"],
               palette=palettes[i], err_kws={'linewidth':1,},
               order=sex_order, alpha=0.5, ax=ax, errorbar='se', capsize=.12, legend=False)
    sb.stripplot(data=sub, x='sex', y='proportion', hue='session', hue_order=["R1", "R2"],
                 palette=palettes[i], order=sex_order, dodge=True, jitter=0, ax=ax, legend=False,
                 edgecolor='k', size=3, alpha=.7, linewidth=.5)
    print(btype)
    print(pg.mixed_anova(data=sub,dv='proportion',
                         between='sex',within='session',subject='animal'))
    print(pg.pairwise_tests(data=sub,dv='proportion',padjust='fdr_bh',within_first=False,
                         between='sex',within='session',subject='animal'))
    for animal, animal_grp in sub.groupby("animal"):
        sex = animal_grp["sex"].iloc[0]
        r1 = animal_grp[animal_grp["session"] == "R1"]["proportion"].values
        r2 = animal_grp[animal_grp["session"] == "R2"]["proportion"].values
        if len(r1) and len(r2):
            x = sex_to_x[sex]
            ax.plot(
                [x + session_xoff["R1"], x + session_xoff["R2"]],
                [r1[0], r2[0]],
                color="gray", alpha=0.5, linewidth=0.8, zorder=0
            )
    ax.set_ylabel('Proportion Time Interacting',size=6)
    ax.set_title(btype,size=7)
    ax.set_xlabel("")
    ax.tick_params(axis='both', labelsize=5,width=.5)
    sb.despine()
    for sp in ax.spines.values():
        sp.set_linewidth(.5)

plt.tight_layout(pad=0.1,h_pad=0.1)
plt.savefig('/Users/amonast/Desktop/miniscope/figures/behavior_barplots.svg',transparent=True)

# everybody all 3 behaviors Proportion time

In [ ]:
BTYPES = ["Social Avoidance", "Pro-Social", "Aggressive"]
session_order = ["R1", "R2"]
session_to_x = {s: i for i, s in enumerate(session_order)}
agg_palette   = ["#fe8382","#fe0806",]
social_palette   = ["#aacffd","#55a0fb"]
avoidance_palette    = ["#dbaaea","#b856d7"]
palettes = [avoidance_palette,social_palette,agg_palette]
height=1.5
width=2.6

fig, axes = plt.subplots(nrows=1, ncols=3, dpi=200, figsize=(width, height), sharey=False)

for i,(ax, btype) in enumerate(zip(axes, BTYPES)):
    sub = duration_summary[duration_summary["behavior_type"] == btype]
    if btype == "Aggressive":
        sub = sub[~sub["animal"].isin(EXCLUDE_AGG)]
    sb.barplot(data=sub, x='session', y='proportion', hue='session',palette=palettes[i],
               err_kws={'linewidth':1,},
               order=session_order, alpha=0.5, ax=ax, errorbar='se', capsize=.12,legend=False)
    sb.stripplot(data=sub, x='session', y='proportion', hue='session',palette=palettes[i],
                 order=session_order, jitter=0, ax=ax,legend=False,
                 edgecolor='k', size=3,alpha=.7,linewidth=.5)
    print(btype)
    print(pg.pairwise_tests(data=sub,dv='proportion',padjust='fdr_bh',
                         within='session',subject='animal'))
    for animal, animal_grp in sub.groupby("animal"):
        r1 = animal_grp[animal_grp["session"] == "R1"]["proportion"].values
        r2 = animal_grp[animal_grp["session"] == "R2"]["proportion"].values
        if len(r1) and len(r2):
            ax.plot(
                [session_to_x["R1"], session_to_x["R2"]],
                [r1[0], r2[0]],
                color="gray", alpha=0.5, linewidth=0.8, zorder=0
            )
    ax.set_ylabel('Proportion Time Interacting',size=6)
    ax.set_title(btype,size=7)
    ax.set_xlabel("")
    ax.tick_params(axis='both', labelsize=5,width=.5)
    sb.despine()
    for sp in ax.spines.values():
        sp.set_linewidth(.5)

plt.tight_layout(pad=0.1,h_pad=0.1)
plt.savefig('/Users/amonast/Desktop/miniscope/figures/behavior_barplots_percent.svg',transparent=True)

# Everybody, all 3 behaviors total time


In [ ]:
BTYPES = ["Social Avoidance", "Pro-Social", "Aggressive"]
session_order = ["R1", "R2"]
session_to_x = {s: i for i, s in enumerate(session_order)}
agg_palette   = ["#fe8382","#fe0806",]
social_palette   = ["#aacffd","#55a0fb"]
avoidance_palette    = ["#dbaaea","#b856d7"]
palettes = [avoidance_palette,social_palette,agg_palette]
height=1.5
width=2.6
fig, axes = plt.subplots(nrows=1, ncols=3, dpi=200, figsize=(width, height), sharey=False)

for i,(ax, btype) in enumerate(zip(axes, BTYPES)):
    sub = duration_summary[duration_summary["behavior_type"] == btype]
    if btype == "Aggressive":
        sub = sub[~sub["animal"].isin(EXCLUDE_AGG)]
    sb.barplot(data=sub, x='session', y='duration', hue='session',palette=palettes[i],
               err_kws={'linewidth':1,},
               order=session_order, alpha=0.5, ax=ax, errorbar='se', capsize=.12,legend=False)
    sb.stripplot(data=sub, x='session', y='duration', hue='session',palette=palettes[i],
                 order=session_order, jitter=0, ax=ax,legend=False,
                 edgecolor='k', size=3,alpha=.7,linewidth=.5)
    print(btype)
    print(pg.pairwise_tests(data=sub,dv='duration',padjust='fdr_bh',
                         within='session',subject='animal'))
    for animal, animal_grp in sub.groupby("animal"):
        r1 = animal_grp[animal_grp["session"] == "R1"]["duration"].values
        r2 = animal_grp[animal_grp["session"] == "R2"]["duration"].values
        if len(r1) and len(r2):
            ax.plot(
                [session_to_x["R1"], session_to_x["R2"]],
                [r1[0], r2[0]],
                color="gray", alpha=0.5, linewidth=0.8, zorder=0
            )
    ax.set_ylabel('Total time (s)',size=6)
    ax.set_title(btype,size=7)
    ax.set_xlabel("")
    ax.tick_params(axis='both', labelsize=5,width=.5)
    sb.despine()
    for sp in ax.spines.values():
        sp.set_linewidth(.5)

plt.tight_layout(pad=0.1,h_pad=0.1)
plt.savefig('/Users/amonast/Desktop/miniscope/figures/behavior_barplots_total_time.svg',transparent=True)

# R1 R2 collapsed

In [ ]:
from matplotlib.lines import Line2D

BTYPES = ["Social Avoidance", "Pro-Social", "Aggressive"]
session_order = ["R1", "R2"]
colors  = [avoidance_palette[1], social_palette[1], agg_palette[1]]   # purple, blue, red
markers = {"R1": "o", "R2": "^"}
height = 2
width  = 2

plot_df = duration_summary[duration_summary["behavior_type"].isin(BTYPES)]
plot_df = plot_df[~((plot_df["behavior_type"] == "Aggressive") & plot_df["animal"].isin(EXCLUDE_AGG))]

fig, ax = plt.subplots(dpi=200, figsize=(width, height))

sb.barplot(data=plot_df, x='behavior_type', y='duration', hue='behavior_type',
           order=BTYPES, hue_order=BTYPES, palette=colors,
           err_kws={'linewidth': 1}, alpha=0.5, errorbar='se', capsize=.12,
           ax=ax, legend=False)

for s in session_order:
    sb.stripplot(data=plot_df[plot_df["session"] == s], x='behavior_type', y='duration',
                 hue='behavior_type', order=BTYPES, hue_order=BTYPES, palette=colors,
                 marker=markers[s], jitter=.15, size=3, alpha=.7,
                 edgecolor='k', linewidth=.5, ax=ax, legend=False)

handles = [Line2D([], [], marker=markers[s], linestyle='', markerfacecolor='w',
                  markeredgecolor='k', markeredgewidth=.5, markersize=3, label=s)
           for s in session_order]
ax.legend(handles=handles, fontsize=5, frameon=False, loc='upper right', handletextpad=0.1)

ax.set_ylabel('Total time (s)', size=6)
ax.set_xlabel("")
ax.tick_params(axis='both', labelsize=5, width=.5)
ax.tick_params(axis='x', rotation=30)
sb.despine()
for sp in ax.spines.values():
    sp.set_linewidth(.5)

plt.tight_layout(pad=0.1)
plt.savefig('figures/behavior_barplots_total_time_r1_r2_both.pdf', transparent=True)

# only avoidance and prosocial

In [ ]:
BTYPES = ["Social Avoidance", "Pro-Social"]
session_order = ["R1", "R2"]
session_to_x = {s: i for i, s in enumerate(session_order)}
agg_palette   = ["#fe8382","#fe0806",]
social_palette   = ["#aacffd","#55a0fb"]
avoidance_palette    = ["#dbaaea","#b856d7"]
palettes = [avoidance_palette,social_palette,agg_palette]
height=1
width=2
fig, axes = plt.subplots(nrows=1, ncols=2, dpi=200, figsize=(width, height), sharey=False)

for i,(ax, btype) in enumerate(zip(axes, BTYPES)):
    sub = duration_summary[duration_summary["behavior_type"] == btype]
    
    sb.barplot(data=sub, x='session', y='duration', hue='session',palette=palettes[i],
               err_kws={'linewidth':1,},
               order=session_order, alpha=0.5, ax=ax, errorbar='se', capsize=.12,legend=False)
    sb.stripplot(data=sub, x='session', y='duration', hue='session',palette=palettes[i],
                 order=session_order, jitter=0, ax=ax,legend=False,
                 edgecolor='k', size=3,alpha=.7,linewidth=.5)
    print(btype)
    print(pg.pairwise_tests(data=sub,dv='duration',padjust='fdr_bh',
                         within='session',subject='animal'))
    for animal, animal_grp in sub.groupby("animal"):
        r1 = animal_grp[animal_grp["session"] == "R1"]["duration"].values
        r2 = animal_grp[animal_grp["session"] == "R2"]["duration"].values
        if len(r1) and len(r2):
            ax.plot(
                [session_to_x["R1"], session_to_x["R2"]],
                [r1[0], r2[0]],
                color="gray", alpha=0.5, linewidth=0.8, zorder=0
            )
    ax.set_ylabel('Time Interacting (s)',size=6)
    ax.set_title(btype,size=7)
    ax.set_xlabel("")
    ax.tick_params(axis='both', labelsize=5,width=.5)
    sb.despine()
    for sp in ax.spines.values():
        sp.set_linewidth(.5)

plt.tight_layout(pad=0.1,h_pad=0.1)
plt.savefig('/Users/amonast/Desktop/miniscope/figures/behavior_barplots_time.svg',transparent=True)

# Aggressive animals only, 

In [ ]:
BTYPES = ["Social Avoidance", "Pro-Social", "Aggressive"]
session_order = ["R1", "R2"]
session_to_x = {s: i for i, s in enumerate(session_order)}
agg_palette   = ["#fe8382","#fe0806",]
social_palette   = ["#aacffd","#55a0fb"]
avoidance_palette    = ["#dbaaea","#b856d7"]
palettes = [avoidance_palette,social_palette,agg_palette]
height=1.5
width=2.6


fig, axes = plt.subplots(nrows=1, ncols=3, dpi=200, figsize=(width, height), sharey=False)
    
for i,(ax, btype) in enumerate(zip(axes, BTYPES)):
    sub = duration_summary[(duration_summary["behavior_type"] == btype) & (duration_summary["animal"].isin(agg_animals))]
    
    if btype == "Aggressive":
        sub = sub[~sub["animal"].isin(EXCLUDE_AGG)]
    sb.barplot(data=sub, x='session', y='duration', hue='session',palette=palettes[i],
               err_kws={'linewidth':1,},
               order=session_order, alpha=0.5, ax=ax, errorbar='se', capsize=.12,legend=False)
    sb.stripplot(data=sub, x='session', y='duration', hue='session',palette=palettes[i],
                 order=session_order, jitter=0, ax=ax,legend=False,
                 edgecolor='k', size=3,alpha=.7,linewidth=.5)
    print(btype)
    print(pg.pairwise_tests(data=sub,dv='duration',padjust='fdr_bh',
                         within='session',subject='animal'))
    for animal, animal_grp in sub.groupby("animal"):
        r1 = animal_grp[animal_grp["session"] == "R1"]["duration"].values
        r2 = animal_grp[animal_grp["session"] == "R2"]["duration"].values
        if len(r1) and len(r2):
            ax.plot(
                [session_to_x["R1"], session_to_x["R2"]],
                [r1[0], r2[0]],
                color="gray", alpha=0.5, linewidth=0.8, zorder=0
            )
    ax.set_ylabel('Time Interacting (s)',size=6)
    ax.set_title(btype,size=7)
    ax.set_xlabel("")
    ax.tick_params(axis='both', labelsize=5,width=.5)
    sb.despine()
    for sp in ax.spines.values():
        sp.set_linewidth(.5)

plt.tight_layout(pad=0.1,h_pad=0.1)
plt.savefig('/Users/amonast/Desktop/miniscope/figures/behavior_barplots_time_agganimals.svg',transparent=True)

In [ ]:
agg_animals

# Bout histograms

In [ ]:
# ── 4-category color map ──────────────────────────────────────────────────────
color_map_categories = {
    'Aggressive':        "#C54A4A",
    'Pro-Social':        "#57A35B",
    'Social Avoidance':  '#7B1FA2',
    'Other':             '#78909C',
    
}

In [ ]:
CATEGORIES = ["Aggressive", "Pro-Social", "Social Avoidance"]

plot_df = intervals[intervals["behavior_type"].isin(CATEGORIES)]

for (animal, session), grp in plot_df.groupby(["animal", "session"]):
    fig, ax = plt.subplots(figsize=(6, 3))

    sb.histplot(
        data=grp,
        x="duration",
        hue="behavior_type",
        hue_order=CATEGORIES,
        palette=color_map_categories,
        element="step",
        stat="count",
        common_norm=False,
        ax=ax
    )

    ax.set_title(f"{animal} {session}")
    ax.set_xlabel("Bout duration (s)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()
